# HDAT DS 독립 모의실습 — Process 8단계 (Starter)

> 현대자동차그룹 또는 현대엔지비의 공식·복원 문제가 아닌 **독립 창작 연습문제**입니다. PyTorch만 사용합니다.

작은 이진분류 파이프라인을 8개의 계약(contract)으로 완성합니다. 각 TODO 아래의 `assert`가 요구사항입니다. 셀을 순서대로 실행하고, 막히면 오류 메시지에서 실제 shape와 dtype을 먼저 확인하세요.

## 실행 환경과 시험 습관

- 권장: Python 3.10–3.12, PyTorch 2.2 이상, JupyterLab/Notebook 7 이상
- 이 노트북은 CPU에서 수 초 안에 실행되도록 작게 만들었습니다.
- 시작 전 **Kernel → Restart Kernel and Clear Outputs**, 큰 단계가 끝날 때마다 **Ctrl/Cmd+S**를 누르세요.
- 제출 직전에는 위에서 아래로 한 번 더 실행해 숨은 상태와 이전 변수 의존성을 없애세요.
- Starter는 TODO가 남아 있어 처음에는 일부 `assert`가 실패하는 것이 정상입니다.

In [ ]:
import platform
import random
import tempfile
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

print('Python :', platform.python_version(), '(권장 3.10–3.12)')
print('PyTorch:', torch.__version__, '(권장 2.2+)')
SEED = 2026
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
device = torch.device('cpu')
print('device :', device)

## 문제 데이터

센서 특징 4개로 정상(0)/이상(1)을 예측하는 작은 합성 데이터입니다. 원본은 일부러 `float64`, 레이블은 1차원으로 제공합니다. 이후 과제에서 모델 입력 계약에 맞추세요.

In [ ]:
g = torch.Generator().manual_seed(SEED)
raw_X = torch.randn(40, 4, generator=g, dtype=torch.float64)
signal = 1.4 * raw_X[:, 0] - 0.9 * raw_X[:, 1] + 0.5 * raw_X[:, 2]
raw_y = (signal + 0.25 * torch.randn(40, generator=g) > 0).to(torch.int64)
print('raw_X:', raw_X.shape, raw_X.dtype)
print('raw_y:', raw_y.shape, raw_y.dtype, 'positive rate=', raw_y.float().mean().item())

## 1. 텐서 계약 맞추기

`X`는 `[N, F]` float32, `y`는 `[N, 1]` float32가 되어야 합니다. `unsqueeze`를 어디에 적용할지 생각하세요.

In [ ]:
# TODO 1: raw_X, raw_y를 복사·변환하세요.
X = None
y = None

assert isinstance(X, torch.Tensor) and isinstance(y, torch.Tensor)
assert X.shape == (40, 4) and y.shape == (40, 1)
assert X.dtype == torch.float32 and y.dtype == torch.float32

## 2. 순서를 보존한 train/validation 분할

앞 32개를 train, 뒤 8개를 validation으로 사용하세요. 여기서는 문제의 계약상 섞지 않습니다.

In [ ]:
# TODO 2: 슬라이싱으로 네 텐서를 만드세요.
X_train = y_train = X_val = y_val = None

assert X_train.shape == (32, 4) and y_train.shape == (32, 1)
assert X_val.shape == (8, 4) and y_val.shape == (8, 1)
assert torch.equal(X_train, X[:32]) and torch.equal(X_val, X[32:])

## 3. 데이터 누수 없는 표준화

평균과 표준편차는 train에서만 계산하고 validation에도 같은 값을 적용하세요. 표준편차가 0일 때를 대비해 작은 하한을 둡니다.

In [ ]:
# TODO 3: keepdim=True, unbiased=False를 사용해 train 통계로 표준화하세요.
mu = sigma = X_train_z = X_val_z = None

assert mu.shape == (1, 4) and sigma.shape == (1, 4)
assert X_train_z.shape == X_train.shape and X_val_z.shape == X_val.shape
assert torch.all(sigma > 0)
assert torch.allclose(X_train_z.mean(0), torch.zeros(4), atol=1e-5)

## 4. Dataset과 DataLoader

train은 batch size 8과 재현 가능한 shuffle, validation은 batch size 8과 `shuffle=False`를 사용하세요.

In [ ]:
# TODO 4: TensorDataset과 DataLoader를 만드세요.
train_ds = val_ds = train_loader = val_loader = None

xb, yb = next(iter(train_loader))
assert xb.shape == (8, 4) and yb.shape == (8, 1)
assert xb.dtype == torch.float32 and yb.dtype == torch.float32
assert len(train_ds) == 32 and len(val_ds) == 8

## 5. 작은 `nn.Module`

4개 특징을 받아 은닉층 8개와 ReLU를 거쳐 logit 1개를 반환하세요. 마지막에 sigmoid를 넣지 않습니다.

In [ ]:
# TODO 5: TinyBinaryClassifier를 완성하세요.
class TinyBinaryClassifier(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        raise NotImplementedError('TODO: 레이어를 정의하세요')

    def forward(self, x):
        raise NotImplementedError('TODO: 순전파를 정의하세요')

model = TinyBinaryClassifier(n_features=4).to(device)
test_logits = model(torch.zeros(3, 4))
assert test_logits.shape == (3, 1) and test_logits.dtype == torch.float32

## 6. 한 번의 학습 step

`BCEWithLogitsLoss`와 Adam을 사용해 `zero_grad → forward → loss → backward → step` 순서를 구현하세요.

In [ ]:
# TODO 6: criterion, optimizer를 만들고 한 step을 수행하세요.
criterion = optimizer = loss = None

assert isinstance(loss, torch.Tensor) and loss.ndim == 0
assert torch.isfinite(loss).item()
assert all(p.grad is not None for p in model.parameters() if p.requires_grad)

## 7. 학습 loop와 validation

30 epoch를 학습하고, 검증 때 `model.eval()`과 `torch.inference_mode()`를 사용하세요. logit 0 이상을 class 1로 판단합니다.

In [ ]:
# TODO 7: 학습 loop와 val_accuracy 계산을 구현하세요.
val_accuracy = None

assert isinstance(val_accuracy, float)
assert 0.0 <= val_accuracy <= 1.0
print(f'validation accuracy: {val_accuracy:.3f}')

## 8. `state_dict` 저장·재로딩·예측 동일성

임시 폴더에 체크포인트를 저장하고 새 모델에 불러오세요. 저장한 모델의 validation 확률이 원래 모델과 같아야 합니다. 시험에서는 여기까지 확인한 뒤 Ctrl/Cmd+S 하세요.

In [ ]:
# TODO 8: state_dict를 저장하고 새 모델에 load_state_dict 하세요.
tmp_dir = tempfile.TemporaryDirectory()
checkpoint_path = Path(tmp_dir.name) / 'tiny_model.pt'
reloaded = None
original_prob = reloaded_prob = None

assert checkpoint_path.exists()
assert isinstance(reloaded, TinyBinaryClassifier)
assert original_prob.shape == (8, 1)
assert torch.allclose(original_prob, reloaded_prob, atol=1e-7)
print('8개 계약 완료 — 저장 후 Kernel Restart + Run All로 다시 확인하세요.')

## 최종 점검표

- [ ] 모든 TODO를 채웠다.
- [ ] shape와 dtype assert를 전부 통과했다.
- [ ] validation에서 `eval`과 inference/no-grad 문맥을 사용했다.
- [ ] 모델 전체가 아니라 `state_dict`를 저장하고 새 객체에 재로딩했다.
- [ ] Kernel Restart + Run All 후 저장했다.